In [1]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_abandon_info_fixed(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        abandon_start = None
        abandon_end = None
        
        # track 요소들을 순회하면서 abandon_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'abandon_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    abandon_start = int(box.get('frame'))
            
            elif label == 'abandon_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    abandon_end = int(box.get('frame'))
        
        return abandon_start, abandon_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_abandon_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 abandon 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 abandon 구간 추출 시작...")
    
    total_abandon_frames = 0
    videos_with_abandon = 0
    videos_without_abandon = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # abandon 정보 추출 (수정된 함수)
        abandon_start, abandon_end = parse_abandon_info_fixed(xml_path)
        
        if abandon_start is None or abandon_end is None:
            videos_without_abandon += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   abandon 구간: {abandon_start} ~ {abandon_end}")
        
        # abandon 구간 유효성 검사
        if abandon_end >= total_frames:
            print(f"   ⚠️ abandon_end({abandon_end})가 총 프레임({total_frames})보다 큼")
            abandon_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # abandon 구간에 있는 프레임만 저장
            if abandon_start <= frame_count <= abandon_end:
                # 파일명: 비디오이름_프레임번호_abandon.jpg
                output_filename = f"{video_name}_{frame_count:03d}_abandon.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_abandon += 1
            total_abandon_frames += saved_frames
            print(f"   ✅ {saved_frames}개 abandon 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   abandon 있는 비디오: {videos_with_abandon}개")
    print(f"   abandon 없는 비디오: {videos_without_abandon}개") 
    print(f"   총 abandon 프레임: {total_abandon_frames}개")
    
    return total_abandon_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(abandon가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 abandon 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        abandon_start,abandon_end = parse_abandon_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (abandon 구간이 아닌 곳)
            is_normal = True
            if abandon_start is not None and abandon_end is not None:
                if abandon_start <= frame_count <= abandon_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and abandon_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (abandon + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/dumping_behavior/train/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/dumping_behavior/train/label"
abandon_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/abandon/train/images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/abandon/normal/train/images"

print("🚀 abandon 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: abandon 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: abandon 구간 추출")
abandon_frames = extract_abandon_frames_fixed(video_dir, xml_dir, abandon_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 abandonn 프레임 : {abandon_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {abandon_frames + normal_frames:,}개")
if abandon_frames > 0:
    print(f"   ⚖️ 비율 (normal:abandon): {normal_frames // abandon_frames}:1")
print("="*50)

🚀 abandon 감지용 데이터셋 구성 시작!

📍 1단계: abandon 구간 추출
총 642개 비디오에서 abandon 구간 추출 시작...


비디오 처리:   0%|          | 0/642 [00:00<?, ?it/s]


📹 C_3_11_6_BU_SMA_08-30_14-30-48_CA_RGB_DF2_M1_F1.mp4
   abandon 구간: 102 ~ 117


비디오 처리:   0%|          | 1/642 [00:03<37:48,  3.54s/it]

   ✅ 16개 abandon 프레임 저장

📹 C_3_11_21_BU_SMA_09-27_12-49-17_CD_RGB_DF2_M3_F3.mp4
   abandon 구간: 128 ~ 146


...

   ✅ C_3_11_18_BU_SMB_09-01_13-44-09_CC_RGB_DF2_M2_F2.mp4: 16개 정상 프레임


정상 구간 처리: 100%|█████████▉| 641/642 [24:48<00:00,  1.23it/s]

   ✅ C_3_11_25_BU_SMA_09-27_13-00-16_CD_RGB_DF2_M3_F3.mp4: 17개 정상 프레임


정상 구간 처리: 100%|██████████| 642/642 [24:48<00:00,  2.32s/it]

   ✅ C_3_11_11_BU_SYB_09-28_13-47-03_CC_RGB_DF2_M2_F2.mp4: 17개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 10157개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 abandonn 프레임 : 15,185개
   🟢 Normal 프레임: 10,157개
   📈 총 프레임: 25,342개
   ⚖️ 비율 (normal:abandon): 0:1


In [2]:
import cv2
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm

def parse_abandon_info_fixed(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        
        abandon_start = None
        abandon_end = None
        
        # track 요소들을 순회하면서 abandon_start/end 찾기
        for track in root.findall('.//track'):
            label = track.get('label')
            
            if label == 'abandon_start':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    abandon_start = int(box.get('frame'))
            
            elif label == 'abandon_end':
                # 첫 번째 box의 frame 속성
                box = track.find('box')
                if box is not None:
                    abandon_end = int(box.get('frame'))
        
        return abandon_start, abandon_end
    
    except Exception as e:
        print(f"❌ XML 파싱 오류 ({xml_file}): {e}")
        return None, None

def extract_abandon_frames_fixed(video_dir, xml_dir, output_dir):
    """수정된 abandon 구간 프레임 추출"""
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 비디오 파일 목록
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    
    print(f"총 {len(video_files)}개 비디오에서 abandon 구간 추출 시작...")
    
    total_abandon_frames = 0
    videos_with_abandon = 0
    videos_without_abandon = 0
    
    for video_file in tqdm(video_files, desc="비디오 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # 대응하는 XML 파일 찾기
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        if not os.path.exists(xml_path):
            print(f"⚠️ XML 파일 없음: {xml_file}")
            continue
        
        # abandon 정보 추출 (수정된 함수)
        abandon_start, abandon_end = parse_abandon_info_fixed(xml_path)
        
        if abandon_start is None or abandon_end is None:
            videos_without_abandon += 1
            continue
        
        # 비디오 열기
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"❌ 비디오 열기 실패: {video_file}")
            continue
        
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        print(f"\n📹 {video_file}")
        print(f"   abandon 구간: {abandon_start} ~ {abandon_end}")
        
        # abandon 구간 유효성 검사
        if abandon_end >= total_frames:
            print(f"   ⚠️ abandon_end({abandon_end})가 총 프레임({total_frames})보다 큼")
            abandon_end = total_frames - 1
        
        frame_count = 0
        saved_frames = 0
        
        # 프레임별로 읽기
        while frame_count < total_frames:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            # abandon 구간에 있는 프레임만 저장
            if abandon_start <= frame_count <= abandon_end:
                # 파일명: 비디오이름_프레임번호_abandon.jpg
                output_filename = f"{video_name}_{frame_count:03d}_abandon.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                # JPG로 저장
                if cv2.imwrite(output_path, frame):
                    saved_frames += 1
            
            frame_count += 1
        
        cap.release()
        
        if saved_frames > 0:
            videos_with_abandon += 1
            total_abandon_frames += saved_frames
            print(f"   ✅ {saved_frames}개 abandon 프레임 저장")
    
    print(f"\n🎉 추출 완료!")
    print(f"   abandon 있는 비디오: {videos_with_abandon}개")
    print(f"   abandon 없는 비디오: {videos_without_abandon}개") 
    print(f"   총 abandon 프레임: {total_abandon_frames}개")
    
    return total_abandon_frames

def extract_normal_frames_fixed(video_dir, xml_dir, output_dir, sampling_rate=10):
    """
    정상 구간(abandon가 아닌 구간) 프레임을 샘플링해서 저장 (수정된 버전)
    sampling_rate: N프레임마다 1개씩 저장 (기본값: 10프레임마다 1개)
    파일명: 비디오이름_프레임번호_normal.jpg
    """
    
    os.makedirs(output_dir, exist_ok=True)
    
    video_files = [f for f in os.listdir(video_dir) if f.endswith('.mp4')]
    total_normal_frames = 0
    
    print(f"총 {len(video_files)}개 비디오에서 정상 구간 추출 시작... ({sampling_rate}프레임당 1개)")
    
    for video_file in tqdm(video_files, desc="정상 구간 처리"):
        video_path = os.path.join(video_dir, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        # XML에서 abandon 구간 정보
        xml_file = video_name + '.xml'
        xml_path = os.path.join(xml_dir, xml_file)
        
        # 수정된 파싱 함수 사용
        abandon_start,abandon_end = parse_abandon_info_fixed(xml_path) if os.path.exists(xml_path) else (None, None)
        
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
            
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_count = 0
        saved_normal = 0
        
        while frame_count < total_frames:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 정상 구간 판단 (abandon 구간이 아닌 곳)
            is_normal = True
            if abandon_start is not None and abandon_end is not None:
                if abandon_start <= frame_count <= abandon_end:
                    is_normal = False
            
            # 정상 구간이고 샘플링 조건에 맞으면 저장
            if is_normal and frame_count % sampling_rate == 0:
                output_filename = f"{video_name}_{frame_count:03d}_normal.jpg"
                output_path = os.path.join(output_dir, output_filename)
                
                if cv2.imwrite(output_path, frame):
                    saved_normal += 1
            
            frame_count += 1
        
        cap.release()
        total_normal_frames += saved_normal
        
        if saved_normal > 0 and abandon_start is not None:
            print(f"   ✅ {video_file}: {saved_normal}개 정상 프레임")
    
    print(f"\n🎉 정상 구간 추출 완료!")
    print(f"   총 정상 프레임: {total_normal_frames}개")
    
    return total_normal_frames

# =============================================================================
# 완전한 실행 코드 (abandon + normal)
# =============================================================================

# 경로 설정
video_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/dumping_behavior/val/video"
xml_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/videos/dumping_behavior/val/label"
abandon_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/abandon/val/images"
normal_output_dir = "/home/ckim/dev_ws/project_ws/mldl_project/data/frames/abandon/normal/val/images"

print("🚀 절도 감지용 데이터셋 구성 시작!")
print("="*50)

# 1단계: abandon 구간 프레임 추출 (수정된 함수)
print("\n📍 1단계: abandon 구간 추출")
abandon_frames = extract_abandon_frames_fixed(video_dir, xml_dir, abandon_output_dir)

# 2단계: 정상 구간 프레임 추출 (수정된 함수)
print("\n📍 2단계: 정상 구간 추출")
normal_frames = extract_normal_frames_fixed(video_dir, xml_dir, normal_output_dir, sampling_rate=10)

# 최종 결과
print("\n" + "="*50)
print("🎉 데이터셋 구성 완료!")
print(f"📊 최종 결과:")
print(f"   🔴 abandonn 프레임 : {abandon_frames:,}개")
print(f"   🟢 Normal 프레임: {normal_frames:,}개")
print(f"   📈 총 프레임: {abandon_frames + normal_frames:,}개")
if abandon_frames > 0:
    print(f"   ⚖️ 비율 (normal:abandon): {normal_frames // abandon_frames}:1")
print("="*50)

🚀 절도 감지용 데이터셋 구성 시작!

📍 1단계: abandon 구간 추출
총 80개 비디오에서 abandon 구간 추출 시작...


비디오 처리:   0%|          | 0/80 [00:00<?, ?it/s]


📹 C_3_11_37_BU_DYB_10-16_14-39-23_CB_RGB_DF2_F1_M1.mp4
   abandon 구간: 132 ~ 153


비디오 처리:   1%|▏         | 1/80 [00:01<01:33,  1.18s/it]

   ✅ 22개 abandon 프레임 저장

📹 C_3_11_36_BU_SMA_09-05_15-33-26_CC_RGB_DF2_M4_F4.mp4
   abandon 구간: 149 ~ 167


...

   ✅ C_3_11_35_BU_SMA_09-05_15-30-21_CA_RGB_DF2_M4_F4.mp4: 16개 정상 프레임


정상 구간 처리:  99%|█████████▉| 79/80 [01:00<00:00,  1.39it/s]

   ✅ C_3_11_38_BU_DYA_07-29_15-53-58_CE_RGB_DF2_F2.mp4: 14개 정상 프레임


정상 구간 처리: 100%|██████████| 80/80 [01:00<00:00,  1.31it/s]

   ✅ C_3_11_35_BU_DYA_08-10_16-35-49_CC_RGB_DF2_M2_F2.mp4: 15개 정상 프레임

🎉 정상 구간 추출 완료!
   총 정상 프레임: 1268개

🎉 데이터셋 구성 완료!
📊 최종 결과:
   🔴 abandonn 프레임 : 1,926개
   🟢 Normal 프레임: 1,268개
   📈 총 프레임: 3,194개
   ⚖️ 비율 (normal:abandon): 0:1
